# Reddit Logs to YTMusic Playlists

In [10]:
# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'
# Path to .tsv reddit search logs filtered to just new entries
reddit_log_path = '..\\..\\reddit-scraper\\db'
search_db_path = '..\\..\\reddit-scraper\\logs'

In [4]:
import os
import glob
import time

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

## YTMusic API and Functions

In [5]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))
yt_res_cache = {}
yt_unmatched_cache = {}

## String Helper Functions

In [6]:
def scrub_title(title):
    orig_title = title
    title = str(title).lower().strip()
    title = unicodedata.normalize('NFKD', title).encode('ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    title = title.replace('| ', '(')
    # remove stuff at end of title
    for k in [
         'official video', 'music video', 'live video','lyric video', 'cover)', 'video)', 'prod.', 'produced by', 
         'album stream', 'album review', 'album version', 'full album','produced by', 'npr music tiny desk concert',
         'anniversary expanded edition']:
        if k in title:
            new_t = title.split(k)[0]
            if len(new_t) > 5:
                title = title.split(k)[0]
    start_char = ['[', '('] 
    for k in [
        'official', 'unoffical', 'free', 'explicit', 'video', 'music', 'nsfw', 'original', 'lyric', 'studio', 'vinyl',
        'full', 'album)', 'audio)', 'cover)', 'convert', 'thissongissick', 'duploc', 'prod', 'leak', 'from', 'lofi hip', 
        'remaster', 'uncensored', '720p', '1080p', '320k' 'repackag', '19', '20', 'dir', 'quality upgrade', 'complete', 
        'with lyrics', 'visualizer', 'deluxe', 'anniversary edition']:
        for s in start_char:
            t = s + k
            if t in title:
                new_t = title.split(t)[0]
                if len(new_t) > 5:
                    title = title.split(t)[0]
    for s in start_char:
        if title.endswith(s):
            title = title[:-1]     
    # remove tokens from title
    for k in ['[hd]', '[hq]', 'hd', 'hq', '()', '[]', ' | ', '{}']:
        if k in title:
            title=title.replace(k, '')
    if title.endswith(' - '):
        title = title[0:-3]
    return title
    

def check_album_scrub_title(title):
    # check for album
    title = title.lower().strip()
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True

    title = scrub_title(title)
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## Parse Reddit .tsv and query YTMusic for Match

* Now checks db tsv to see if ialready matched (basd on sub and url)
* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 12/29/2021


In [9]:
db_tsv_path = os.path.join(search_db_path, 'ytmusic', 'reddit_ytmusic_subreddit_db.tsv')
db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)
db['reddit_post_id'] = db.reddit_sub + '//' + db.reddit_source_url
print(f'Loaded {len(db)} entries from {db_tsv_path}')
prev_match_ids = frozenset(db['reddit_post_id'])


Loaded 20823 entries from ..\..\reddit-scraper\logs\ytmusic\reddit_ytmusic_subreddit_db.tsv


In [12]:
reddit_subfolders = ['new']
log_tsvs = []
for folder in reddit_subfolders:
    tsv_path = os.path.join(reddit_log_path, folder)
    log_tsvs += list(glob.glob(os.path.join(tsv_path, '*.tsv')))
log_tsvs = sorted(log_tsvs)
print(f'Found {len(log_tsvs)} reddit tsvs')

log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 2
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']

matched_entries = []
unmatched_entries = []
for i, tsv_file in enumerate(log_tsvs):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'\n\n({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = check_album_scrub_title(entry.title)
        
        # If already in db get match from there
        post_id = f'{sub}//{entry.url}'
        if post_id in prev_match_ids:
            continue
        # Check cache for saved YTMusic query response or previous match failures        
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_album', ''))}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_title', ''))}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_post_id'] = post_id
                unmatch['reddit_sub_id'] = f'{sub}//{entry.url}'
                unmatch['reddit_aggregator'] = agg
                unmatch['manual_label'] = 'no-match'
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            # match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_post_id'] = post_id
        match['reddit_sub_id'] = f'{sub}//{entry.url}'
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id
        match['manual_label'] = f'no-label_{date.today()}'

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Using manual grading to set thresholds
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 60:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 75:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')


        # Update match results
        matched_entries.append(match)

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {search_db_path}')
match_file = os.path.join(search_db_path, 'ytmusic',
                          f'reddit_2021-new_ytmusic_scored_new_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    search_db_path, 'ytmusic', f'reddit_2021-new_ytmusic_failed_new_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)


Found 94 reddit tsvs


(0/94)  Loaded 909 entries with 14 columns from 2000smusic_all
Skipping : 50 Cent - P.I.M.P. (Ziggy Remix) [CBR XCLSV Video]
Skipping : Eddie Vedder w/ Tom Petty & The Heartbreakers - The Waiting - 7.03.06 -1080.HD


(1/94)  Loaded 965 entries with 14 columns from 2010smusic_all
Skipping : cheap thrills - sia - drum overdub cover - fantasia style improvisation
Skipping : Madeon - Pay No Mind (ft  Passion Pit) (BBC1 Radio Rip 2/9/2015)
Skipping : Ed Sheeran - Shape Of You [UT99 - Unreal Tournament]
Skipping : summer - calvin harris - drum overdub cover - fantasia style improvisation
Skipping : Lydia Lunch & Cypress Grove - Blaze Of Glory
Skipping : 22 - taylor swift - drum overdub cover - fantasia style improvisation


(2/94)  Loaded 845 entries with 14 columns from 50sMusic_all
Skipping : The Way I Walk - Jack Scott on Carlton 78 RPM shellac (1959)
Skipping : Okeefenokee (The Cellos - The Juicy Crocodile)
Skipping : 1955 Diamonds - Nip Sip
Skipping : Susie Darlin

In [13]:
db_reddit_keys = frozenset(db.reddit_key)
manual_labels  = []
new = 0
for r in match_df.itertuples():
    if r.reddit_key in db_reddit_keys:
        db_entries = db.loc[db.reddit_key == r.reddit_key]
        if len(db_entries.manual_label.unique()) > 1:
            print('\nWARNING: different manual labels, same key', r.ytmusic_key)
            print(db_entries[['ytmusic_key', 'reddit_key', 'manual_label']])
        manual_labels.append(db_entries.iloc[0].manual_label) 
    else:
        manual_labels.append(f'no-label_{date.today()}')
        new += 1
match_df['manual_label'] = manual_labels
print('need to label:', new)

fixed_match_file = os.path.join(search_db_path, 'ytmusic',
                          f'reddit_2021-new_ytmusic_scored_new_matches_fixed_{date.today()}.tsv')
match_df.to_csv(fixed_match_file, sep='\t', header=True)


                 ytmusic_key              reddit_key  manual_label
20315                    NaN  m1st - kill me gently       no-match
18265  m1st - kill me gently  m1st - kill me gently   passed-match
need to label: 35363


# Grading process (outdated)


## Create YTMusic Playlists from matches

* manually marked each match as 'ok' (pass) or 'x' (fail)
    * see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
* computed match score using a few fuzz algos
    * set basic threshold to try to automate grading

The cells below:
1. Load the graded matches and split into pass / fail matches
2. Merges the failing matches with the unmatched reddit posts, then saves a new tsv with all unmatched 
3. Save the passing tracks to respective subreddit track playlists (split into like / unrated)
4. Save the passing albums to respective subreddit album playlists (split out liked tracks to merge with track playlist)

### Done
* use fuzzy id scoring to determine if match is passing
    * if failing add to unmatched entries
    * if passing add to ytmusic {r.subreddit} playlist
        * have one playlist for albums and another for tracks
* split out 'likes' for each {r.subreddit} ytmusic playlist
* handle passing tacks and albums by adding to subreddit playlist
* handle failed matches, att them to the unmaatched tsv with their scores and extra cols intact
    * maybe retry search? look and see if search query can be done better
    * if source is youtube add youtube link to correct ytmusic sub playlist (last resort)
        * add to albums vs track playist based on duration? 
#### Last run 10/17/2021

## Re-match (round 1)
reddit_scraper\logs\2021_new\PunkRock_top-all_1000_1635049772.tsv
reddit_scraper\logs\2021_new\PunkRock_top-year_1000_1635052086.tsv
reddit_scraper\logs\2021_new\witchHouse_top-year_1000_1635051978.tsv
reddit_scraper\logs\2021_new\witchHouse_top-all_1000_1635049613.tsv

In [ ]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_scored_matches_and_graded_2021-10-16.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']

# Merge failing with unmatched and save new tsv
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_failed_matches_2021-10-16.tsv')
unmatched_df = pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
all_unmatched = pd.concat([failing, unmatched_df]).sort_values('reddit_sub')
print(f'Saving {len(all_unmatched)} unmatched reddit entries')
all_unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_graded_{date.today()}.tsv')
all_unmatched.to_csv(all_unmatch_file, sep='\t', header=True)

## Re-match (round 2)

graded 2021 refresh and more 2019-scraped

In [ ]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_and_2021-refresh_ytmusic_scored_and_graded_matches_2021-10-24.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('idx b')

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId
print(len(graded_matches))
graded_matches = graded_matches.drop_duplicates('ytmusic_playlist_id')
print(len(graded_matches))

graded_matches.loc[graded_matches.is_album==True]

3618
3559


,idx b,idx a,manual_check,ytmusic_key,reddit_title,match_score_token_set_ratio,match_score_token_sort_ratio,match_quality,is_album,ytmusic_album,...,ytmusic_duration,ytmusic_year,ytmusic_resultType,reddit_key,reddit_sub,reddit_aggregator,reddit_file,reddit_source_url,youtube_id,Unnamed: 26
50,24.0,NaN,ok,- illmatic,Illmatic (Full Album),100.0,100.0,1.0,True,Illmatic,...,NaN,1994.0,album,illmatic,90shiphop,top-all,NaN,https://www.youtube.com/watch?v=oqVD2yQqH7M&fe...,oqVD2yQqH7M,Nas
3480,1304.0,NaN,ok,- meridian,Rohne - Meridian [Full Album],100.0,73.0,1.0,True,Meridian,...,NaN,2018.0,album,rohne - meridian,triphop,top-all,NaN,https://www.youtube.com/watch?v=FozmE-RbpC0,FozmE-RbpC0,Rohne
3445,3531.0,NaN,x,- ???????,???? - ???????/Birth of a New Day (Full Album)...,0.0,0.0,0.0,True,???????,...,NaN,2015.0,album,2814 - /birth of a new day,futurebeats,top-year,NaN,https://www.youtube.com/watch?v=F9L4q-0Pi4E,F9L4q-0Pi4E,2814
1857,3785.0,NaN,ok,"- all those estranged things i have created, l...",Grand Inc - All Those Estranged Things I Have ...,100.0,91.0,1.0,True,"All Those Estranged Things I Have Created, Loo...",...,NaN,2021.0,album,grand inc - all those estranged things i have ...,futurebeats,top-year,NaN,https://www.youtube.com/watch?v=-zZToo9_OeE,-zZToo9_OeE,Grand Inc
3202,4341.0,NaN,x,- sunday avenue,DJ Hollow - Nightlife [Lofi House / Garage] (F...,24.0,24.0,0.0,True,Sunday Avenue,...,NaN,2017.0,album,dj hollow - nightlife [lofi house / garage],futuregarage,top-year,NaN,https://www.youtube.com/watch?v=GN8A8lNjnfE&ab...,GN8A8lNjnfE,DJ Boring
3490,5134.0,NaN,ok,- o.s.t.,People Under The Stairs - O.S.T. [Full Album],100.0,29.0,0.0,True,O.S.T.,...,NaN,2002.0,album,people under the stairs - o.s.t.,hiphop,top-year,NaN,https://www.youtube.com/watch?v=RwKlQ9k7Lzk&fe...,RwKlQ9k7Lzk,People Under The Stairs
3444,6512.0,NaN,x,- ?????,MF DOOM X Tatsuro Yamashita (Full Album),0.0,0.0,0.0,True,?????,...,NaN,2016.0,album,mf doom x tatsuro yamashita,jazz,top-year,NaN,https://www.youtube.com/watch?v=bqkOQ46lxj8,bqkOQ46lxj8,???
3010,6532.0,NaN,x,- kofi,"Donald Byrd ?""Kofi"" Full Album",100.0,26.0,0.0,True,Kofi,...,NaN,1971.0,album,"donald byrd ""kofi"" full album",jazz,top-year,NaN,https://www.youtube.com/watch?v=0Ve415qxUbs,0Ve415qxUbs,Donald Byrd
3483,6573.0,NaN,ok,- red clay,FREDDIE HUBBARD - Red Clay LP 1970 Full Album,100.0,31.0,0.0,True,Red Clay,...,NaN,1987.0,album,freddie hubbard - red clay lp 1970 full album,jazz,top-year,NaN,https://www.youtube.com/watch?v=KLc8vFT8y7Q,KLc8vFT8y7Q,Freddie Hubbard
3051,6670.0,NaN,x,- promises,"Floating Points, Pharoah Sanders & The London ...",100.0,21.0,0.0,True,Promises,...,NaN,2021.0,album,"floating points, pharoah sanders & the london ...",jazznoir,top-year,NaN,https://www.youtube.com/watch?v=Mn8x0QbN4f8,Mn8x0QbN4f8,"Floating Points, Pharoah Sanders & The London ..."


In [ ]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_and_2021-refresh_ytmusic_scored_and_graded_matches_2021-10-24.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('idx b')

# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId
print(len(graded_matches))
graded_matches = graded_matches.drop_duplicates('ytmusic_playlist_id')
print(len(graded_matches))

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']


# Merge failing with unmatched and save new tsv
# Prevsioulsy merged unmatch + failed from 10-16
unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_failed_matches_graded_2021-10-16.tsv')
unmatched_df = pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
print('failed to match 10-17: ',len(unmatched_df))

# Failed to match 10-24
failed_match_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_2021-refresh_ytmusic_failed_matches_2021-10-24.tsv')
failed_df = pd.read_csv(failed_match_tsv, sep='\t', index_col=0)
print('failed to match 10-24: ',len(failed_df))

# Failed after grading 10-24
failing_recent = failing.loc[~failing.reddit_file.isna()]
print('failed from grading 10-24: ',len(failing_recent))

# MERGE
all_unmatched = pd.concat([unmatched_df, failing_recent, failed_df]).sort_values('reddit_sub')
print(f'Saving {len(all_unmatched)} unmatched reddit entries')
all_unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_graded_{date.today()}.tsv')
all_unmatched.to_csv(all_unmatch_file, sep='\t', header=True)


3618
3559
failed to match 10-17:  4911
failed to match 10-24:  505
failed from grading 10-24:  316
Saving 5732 unmatched reddit entries


# After Grading...

In [18]:
graded_tsv = os.path.join(search_db_path, 'ytmusic',
                    'reddit_2021-new_ytmusic__round-1-graded_2022-1-1.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('reddit_sub')

passing = graded_matches.loc[graded_matches.manual_label == 'passed-match']
failing = graded_matches.loc[graded_matches.manual_label == 'failed-match']
print(f'{graded_matches.shape} shaped graded matches, {len(passing)} passing, {len(failing)} failing')

(8404, 25) shaped graded matches, 7439 passing, 965 failing


# Grading First round (1/1/22)

## Subreddit playlists for passing tracks

(same code for round 1 and 2, just clear completed [])

In [26]:
N=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == 'FALSE']
completed = []
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing tracks')
    vids = df.ytmusic_videoId.unique().tolist()
    title=f'x_r.{sub}_tracks'
    desc = f'Matched {len(vids)} tracks from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} tracks playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    if len(tracks) < 10:
        print(f'Not enough tracks to split into like dislike: count = {len(tracks)}')
        continue

    # Create Like subset
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)
    
    # Create Unrated Radio subset
    unrated_tracks = tracks.loc[tracks['likeStatus'] != 'LIKE']
    vids = unrated_tracks.videoId.unique().tolist()
    desc = f'Unrated radio subset of {len(vids)} entries from: {desc}'
    radio_pl_id = ytm.create_playlist(title=f'{title}_radio', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} INDIFFERENT r.{sub} radio tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Delete original playlist now thet like/unrated is split
    ytm.delete_playlist(pl_id)
    print(f'Deleted r/{sub} tracks playlist with id: {pl_id}')
    completed.append(sub)
    



Generating r.2000smusic ytmusic playlist for 828 passing tracks
Saved 739 r.2000smusic tracks playlist with id: PLWptjpDqazOwzpr9NxwEuUS8q2FP42dFv, waiting 5 seconds...
Filtered 71 LIKE r.2000smusic tracks playlist with id: PLWptjpDqazOxewqoazfQtfjQmHT4_bW-5, waiting 5 seconds...
Filtered 668 INDIFFERENT r.2000smusic radio tracks playlist with id: PLWptjpDqazOxewqoazfQtfjQmHT4_bW-5, waiting 5 seconds...
Deleted r/2000smusic tracks playlist with id: PLWptjpDqazOwzpr9NxwEuUS8q2FP42dFv

Generating r.2010smusic ytmusic playlist for 802 passing tracks
Saved 792 r.2010smusic tracks playlist with id: PLWptjpDqazOwCIlW2zjgyHzaWXMsYVm3c, waiting 5 seconds...
Filtered 66 LIKE r.2010smusic tracks playlist with id: PLWptjpDqazOwEIzhrS77YuHV9AskcWxiN, waiting 5 seconds...
Filtered 726 INDIFFERENT r.2010smusic radio tracks playlist with id: PLWptjpDqazOwEIzhrS77YuHV9AskcWxiN, waiting 5 seconds...
Deleted r/2010smusic tracks playlist with id: PLWptjpDqazOwCIlW2zjgyHzaWXMsYVm3c

Generating r.50sMusic

## Subreddit playlists for passing albums
(same code for round 1 and 2, just clear completed [])

In [27]:
SLEEP_TIME=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == 'TRUE']
completed= []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']

for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing albums')
    vids = set()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
            vids.add(track['videoId'])
    vids = list(vids)

    title=f'x_r.{sub}_albums'
    desc = f'Matched {len(vids)} albums from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} albums playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_tracks_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)
    completed.append(sub)




Generating r.2000smusic ytmusic playlist for 2 passing albums
 Adding album: From Under The Cork Tree Limited Tour Edition (Limited Tour Edition)
 Adding album: Bulls & The Bees / Electroretard
Saved 31 r.2000smusic albums playlist with id: PLWptjpDqazOxKzhRP1MX_sCma-8XPGzmV, waiting 5 seconds...
Filtered 0 LIKE r.2000smusic tracks playlist with id: PLWptjpDqazOySBtOWUZg7t9fJvdwb5RCf, waiting 5 seconds...

Generating r.60sMusic ytmusic playlist for 2 passing albums
 Adding album: Sounds Of Silence
 Adding album: Philosophy of the World
Saved 23 r.60sMusic albums playlist with id: PLWptjpDqazOyZ20oA36nXh7MzG2MtLC2R, waiting 5 seconds...
Filtered 6 LIKE r.60sMusic tracks playlist with id: PLWptjpDqazOzeixNJyXyKv-s7QgBx2z7g, waiting 5 seconds...

Generating r.70s ytmusic playlist for 3 passing albums
 Adding album: The Slider
 Adding album: Aqualung (40th Anniversary Edition)
 Adding album: Transiberiana (Bonus Tracks Version)
Saved 51 r.70s albums playlist with id: PLWptjpDqazOzsY3vT_tN